In [18]:
import numpy as np 
import math
import torch, torchvision
import torch.nn as nn 
import torch.optim as optim 
import torchvision.transforms as transforms 
from torch.utils.data import DataLoader 
import matplotlib.pyplot as plt 
from dataclasses import dataclass 
import torch.nn.functional as F 

In [2]:
torch.manual_seed(20) 

In [3]:
@dataclass
class TrainConfig:
    block_size: int = 1024
    vocab_size: int = 64 
    n_layers: int = 4
    n_head: int = 8 
    n_embd: int = 120
    dropout: float = 0.2 
    bias: bool = False 

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__() 
        assert config.n_embd % config.n_head == 0 

        # qkv projection 
        self.c_attn = nn.Linear(config.n_embd, config.n_embd*3, bias=config.bias)

        # MHA output projection 
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias) 

        # dropouts 
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout) 

        # save config for convenience
        self.n_embd = config.n_embd
        self.n_head = config.n_head 

        # create a mask matrix in memory
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config.block_size, config.block_size))
                .view(1, 1, config.block_size, config.block_size)
        )

    def forward(self, x):
        # get input dimensions 
        B, T, C = x.shape 

        # qkv projection [B, T, 3C] -> split into [B, T, C] each 
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2) 

        # split across attention heads: [B, T, C] -> [B, T, nh, hs] -> [B, nh, T, hs] 
        q = q.view(B, T, self.n_head, self.n_embd // self.n_head).transpose(1, 2) 
        k = k.view(B, T, self.n_head, self.n_embd // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.n_embd // self.n_head).transpose(1, 2) 

        # attention scores: [B, nh, T, hs] @ [B, nh, hs, T] -> [B, nh, T, T]
        att = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])

        # mask out the upper triangular, as the current token cannot attend future tokens 
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        # contextual emb: [B, nh, T, T] @ [B, nh, T, hs] -> [B, nh, T, hs]
        y = att @ v 

        # assemble outputs: [B, nh, T, hs] -> [B, T, nh, hs] -> [B, T, C]
        y = y.transpose(1, 2).contiguous().view(B, T, C) 

        # output projection -> residual dropout 
        y = self.resid_dropout(self.c_proj(y)) 
        return y 

class LayerNorm(nn.Module):
    def __init__(self, n_embd: int = 64, bias: bool = False):
        super().__init__() 
        self.weight = nn.Parameter(torch.ones(n_embd))
        self.bias = nn.Parameter(torch.ones(n_embd)) if bias else None 

    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5) 

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__() 

        # project up: embd_dim -> emb_dim * 4
        self.c_fc = nn.Linear(config.n_embd, config.n_embd*4, bias=config.bias)
        # non-linear activation
        self.gelu = nn.GELU()
        # project back: embd_dim * 4 -> embd_dim 
        self.c_proj = nn.Linear(config.n_embd*4, config.n_embd, bias=config.bias)
        # dropout
        self.dropout = nn.Dropout(config.dropout) 

    def forward(self, x):
        x = self.c_fc(x) 
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x 

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()

        # layer norm
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        # attention 
        self.attn = CausalSelfAttention(config)
        # layer norm
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        # pointwise ffn
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x)) 
        x = x + self.mlp(self.ln_2(x)) 
        return x 

class Decoder(nn.Module):
    def __init__(self, config):
        super().__init__()

        # embedding tables
        self.tok_emb_table = nn.Embedding(config.vocab_size, config.n_embd) # token embed lookup: [vocab_size, emb_size]
        self.pos_enc_table = nn.Parameter(torch.randn(config.block_size, config.n_embd)*0.02) # learnable positional encodings: [block_size, emb_size]

        # transformer blocks
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)]) 

        # final layer norm
        self.ln_f = LayerNorm(config.n_embd, bias=config.bias)

        # final dropout
        self.dropout = nn.Dropout(config.dropout)

        # language modeling head 
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=config.bias)

        self.config = config 

    def forward(self, x_ids):
        # get input shape 
        B, T = x_ids.shape 
        assert T <= self.config.block_size, f'seq length {T} has exceeded max context window {self.config.block_size}'

        # look up token embs
        tok_emb = self.tok_emb_table(x_ids) # [B, T, C]
        # get first T roles from pos emb
        pos_emb = self.pos_enc_table[:T].unsqueeze(0) # [1, T, C]
        # add together to get input emb
        x = tok_emb + pos_emb # [B, T, C]

        # pass to transformer blocks
        for block in self.blocks:
            x = block(x) 

        x = self.ln_f(x)
        logits = self.lm_head(x) 
        return  logits 

In [17]:
nn.Parameter(torch.ones(4))

Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)

In [9]:
m = torch.tril(torch.ones(4, 4))
m 

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

In [15]:
a = a.masked_fill(m==0, float('-inf'))
a 

tensor([[ 0.4896,    -inf,    -inf,    -inf],
        [ 0.6792, -0.0396,    -inf,    -inf],
        [ 1.1870, -0.2625,  0.9833,    -inf],
        [-0.2575,  1.7480, -1.9582, -0.1159]])